In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
training_set = datasets.FashionMNIST(
    root="../../../data",
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(
            torch.float32,
            scale=True
        )
    ])
)

testing_set = datasets.FashionMNIST(
    root="../../../data",
    train=False,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(
            torch.float32,
            scale=True
        )
    ])
)

In [3]:
# VARS
BATCH_SIZE = 64

# loaders
train_dataloader = DataLoader(
    dataset=training_set, 
    batch_size=BATCH_SIZE
)
test_dataloader = DataLoader(
    dataset=testing_set, 
    batch_size=BATCH_SIZE
)

for X, y in test_dataloader:
    print(f"Shape of X [Number, Channel, Height, Weight]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [Number, Channel, Height, Weight]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type  
else:
    device = "cpu"

print(f"Using {device} device")

Using cpu device


In [5]:
# Define model
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.my_layer_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.my_layer_stack(x)
        return logits

model = NeuralNet().to(device)
print(model)

NeuralNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (my_layer_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), 
    lr=1e-3
)

In [7]:
def train(dataloader, model, loss_fn, optimizer, show_log):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0 and show_log:
            loss = loss.item()
            current = (batch + 1) * len(X)
            print(f"Batch: {batch} | Loss: {loss:.2f}  [{current:.2f}/{size:.2f}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = correct = 0
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print("==============================")
    print(f"Test Error:")
    print("==============================")
    print(f"Accuracy: {(100*correct):.2f}%")
    print(f"Avg loss: {test_loss:.2f}")

In [8]:
epochs = 15
for t in range(epochs):
    show_log = False
    if t % 5 == 0:
        show_log = True

    if show_log:
        print(f"Epoch {t+1}")
        print("-------------------------------")

    train(train_dataloader, model, loss_fn, optimizer, show_log)

    if show_log:
        test(test_dataloader, model, loss_fn)

print("==============================")
print("Done!".center(30))
print("==============================")

Epoch 1
-------------------------------
Batch: 0 | Loss: 2.30  [64.00/60000.00]
Batch: 100 | Loss: 2.30  [6464.00/60000.00]
Batch: 200 | Loss: 2.28  [12864.00/60000.00]
Batch: 300 | Loss: 2.28  [19264.00/60000.00]
Batch: 400 | Loss: 2.25  [25664.00/60000.00]
Batch: 500 | Loss: 2.22  [32064.00/60000.00]
Batch: 600 | Loss: 2.23  [38464.00/60000.00]
Batch: 700 | Loss: 2.20  [44864.00/60000.00]
Batch: 800 | Loss: 2.20  [51264.00/60000.00]
Batch: 900 | Loss: 2.17  [57664.00/60000.00]
Test Error:
Accuracy: 46.88%
Avg loss: 2.17
Epoch 6
-------------------------------
Batch: 0 | Loss: 1.16  [64.00/60000.00]
Batch: 100 | Loss: 1.16  [6464.00/60000.00]
Batch: 200 | Loss: 0.99  [12864.00/60000.00]
Batch: 300 | Loss: 1.12  [19264.00/60000.00]
Batch: 400 | Loss: 0.98  [25664.00/60000.00]
Batch: 500 | Loss: 1.02  [32064.00/60000.00]
Batch: 600 | Loss: 1.05  [38464.00/60000.00]
Batch: 700 | Loss: 0.99  [44864.00/60000.00]
Batch: 800 | Loss: 1.03  [51264.00/60000.00]
Batch: 900 | Loss: 0.96  [57664.0

In [11]:
test(train_dataloader, model, loss_fn)

Test Error:
Accuracy: 76.81%
Avg loss: 0.66
